In [2]:
pip install python-dotenv

In [3]:
from dotenv import load_dotenv
load_dotenv()

False

In [8]:
pip install openai langchain-openai langchain-google-genai langgraph jinja2 json-repair tavily opentelemetry-instrumentation-logging

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.1.2
    Uninstalling wrapt-2.1.2:
      Successfully uninstalled wrapt-2.1.2
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-0.59b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followi

In [5]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool
from uuid import uuid4

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("Simple logging demo")
logger.setLevel(logging.DEBUG)


def check_delivery_status(order_id: str):
    """Check the delivery status based on order ID."""
    logging.info(f"Tool check_delivery_status was called with order_id {order_id}")
    return {
        "order_id": order_id,
        "order_status": "In delivery",
        "order_last_location": "Sedang disortir di DC cakung",
        "ETA": "31-12-2025",
    }


@tool
def check_shopping_cart(user_id: str):
    """Retrieve the user's shopping cart based on user ID."""
    logging.info(f"Tool check_shopping_cart was called with user_id {user_id}")
    return {
        "user_id": user_id,
        "items_in_cart": [
            {
                "item_id": 1,
                "item_name": "babibas running shoes",
                "item_type": "shoes",
                "item_details": {"size": 48, "color": "bright pink"},
            },
            {
                "item_id": 22,
                "item_name": "bortusten performance socks",
                "item_type": "socks",
                "item_details": {"size": "XL", "color": "bright yellow"},
            },
            {
                "item_id": 111,
                "item_name": "becs anti-slip shoe lace",
                "item_type": "shoe_accessories",
                "item_details": {"color": "cyan"},
            },
        ],
    }


SYSTEM_PROMPT = """\
Act as a friendly shopping assistant. You will be provided with tools to help you answer the user's queries.
When a tool is not available to help a user's request, suggest to contact a human instead.
Do not make up answers, if a user does not provide the required information, ask them.
Answer should be in fully natural language. It's okay to add small formatting.
Answer in Indonesian by default unless the user asks in english.
"""

# ganti inisiasi kalau menggunakan provider lain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("API_KEY_LLM")
)
# tidak perlu bingung dulu dengan terminologi "Agent" di sini
# anggap saja LLM biasa dengan tool
agent = create_agent(
    model=llm,
    tools=[check_delivery_status, check_shopping_cart],
    system_prompt=SYSTEM_PROMPT,
)


def chat(query):
    chat_id = uuid4().hex
    logging.info(f"Event: new chat Chat ID: {chat_id} Content: {query}")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
    )
    result = result["messages"][-1].content
    logging.info(f"Event: new response ChatID: {chat_id} Response: {result}")
    return result

# print(">> Pemanggilan 1, perlu tool check status pengiriman <<")
# print(
chat(
    "Halo, saya ingin mengecek status pengiriman pesanan saya dengan ID 1112231233"
)
# )
# print("====== separator ======")
# print(">> Pemanggilan 2, perlu tool check keranjang belanja <<")
# print(
chat("Halo, saya ingin mengecek keranjang belanja saya dengan user ID Ivan#19827")
# )
# print("====== separator ======")
# print(">> Pemanggilan 3, tidak perlu tool apapun, cuma pertanyaan biasa <<")
# print(
chat("Halo, apa kabar? Siapa kamu?")
# )

'Halo! Saya adalah asisten belanja Anda. Saya di sini untuk membantu Anda dengan pertanyaan terkait belanja.'

In [6]:
import json
import logging
from datetime import datetime, UTC

# formatter untuk setiap log
class JSONFormatter(logging.Formatter):
    def format(self, record):
        # 4 data di bawah ini by default selalu ada
        payload = {
            "timestamp": datetime.now(UTC).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
            "logger": record.name,
        }
        # setiap baris log adalah json individual
        return json.dumps(payload)
# inisiasi logger
logger = logging.getLogger("demo")
logger.setLevel(logging.DEBUG)

# mengatur agar handler yang
# mengatur bagaimana log mengirimkan output
# agar menggunakan formatter di atas
handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())

logger.addHandler(handler)

logger.info("ini log info")
logger.debug("ini log debug")
logger.warning("ini log warning")
logger.error("ini log error")

{"timestamp": "2026-04-01T11:16:18.067578+00:00", "level": "INFO", "message": "ini log info", "logger": "demo"}
INFO:demo:ini log info
{"timestamp": "2026-04-01T11:16:18.071847+00:00", "level": "DEBUG", "message": "ini log debug", "logger": "demo"}
DEBUG:demo:ini log debug
{"timestamp": "2026-04-01T11:16:18.074424+00:00", "level": "WARNING", "message": "ini log warning", "logger": "demo"}
{"timestamp": "2026-04-01T11:16:18.076901+00:00", "level": "ERROR", "message": "ini log error", "logger": "demo"}
ERROR:demo:ini log error


In [1]:
import json
import logging
import time
from typing import List, Dict, Any

from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.instrumentation.logging import LoggingInstrumentor


# setup otel, terutama service.name
resource = Resource.create({"service.name": "dummy-rag-service"})
provider = TracerProvider(resource=resource)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

# idealnya ini dihubungkan dengan collector untuk dikirim ke backend OTLP
# tapi untuk demo, di-print ke terminal langsung saja
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# ini metode untuk augmentasi logging python.
# kita tidak perlu repot mengatur sendiri formatnya
LoggingInstrumentor().instrument(set_logging_format=True)
logger = logging.getLogger("dummy_rag")
logger.setLevel(logging.DEBUG)
for handler in logger.handlers:
    logger.removeHandler(handler)

# structured output
class JsonFormatter(logging.Formatter):
    def format(self, record):
        log = {
            "timestamp": self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level": record.levelname,
            "message": record.getMessage(),
            "logger": record.name,
        }
        # include trace/span context
        span = trace.get_current_span()
        ctx = span.get_span_context()
        if ctx.is_valid:
            log["trace_id"] = format(ctx.trace_id, "032x")
            log["span_id"] = format(ctx.span_id, "016x")
        return json.dumps(log)
handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger.addHandler(handler)

def span_event(span, event: str, **attrs):
    span.add_event(event, attributes=attrs)
    # also mirror as structured JSON logs
    logger.info(json.dumps({"event": event,**attrs}))

# retrieval dummy
class DummyVectorStore:

    def __init__(self):
        self.docs = [
            {"id": "doc1", "text": "Jakarta mengalami penurunan tanah karena overextraction of groundwater."},
            {"id": "doc2", "text": "Solusi teknis termasuk managed aquifer recharge dan piping dari sumber permukaan."},
            {"id": "doc3", "text": "Kebijakan publik diperlukan untuk mengatur pemompaan air tanah."},
        ]

    def search(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        tokens = set(query.lower().split())
        scored = []
        for d in self.docs:
            score = sum(1 for t in tokens if t in d["text"].lower())
            scored.append((score, d))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [doc for score, doc in scored[:k]]

# LLM dummy
class DummyLLM:

    def __init__(self, model_name: str = "dummy-llm-0.1"):
        self.model_name = model_name

    def generate(self, prompt: str, max_tokens: int = 128) -> Dict[str, Any]:
        # pretend we "tokenized" the prompt
        token_count = len(prompt.split()) + 10
        # produce a simple response using the prompt
        response = f"[DUMMY GENERATION based on prompt snippet] {prompt[:200]}"
        return {
            "text": response,
            "tokens": token_count,
            "model": self.model_name,
        }


# pipeline RAG dummy
class DummyRAG:
    def __init__(self, store: DummyVectorStore, llm: DummyLLM):
        self.store = store
        self.llm = llm
        self.tracer = tracer

    def ingest(self, docs: List[Dict[str, Any]]):
        # satu span = 1 proses.
        # setiap span perlu di-set manual menggunakan context manager,
        # lalu di-set manual juga atribute (metadata) nya
        with self.tracer.start_as_current_span("ingest_documents") as span:
            span.set_attribute("component", "ingest")
            span.set_attribute("document.count", len(docs))
            span_event(span, "ingest_event", message=(f"Ingesting {len(docs)} documents"))
            time.sleep(0.01)
            span.set_attribute("ingest.status", "ok")

    def retrieve(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        # sama seperti di atas
        # satu span menyimpan keseluruhan informasi proses retrieval
        with self.tracer.start_as_current_span("retrieve_documents") as span:
            span.set_attribute("component", "retrieval")
            span.set_attribute("retrieval.query", query)
            span.set_attribute("retrieval.top_k", k)
            start = time.time()
            results = self.store.search(query, k=k)
            latency_ms = (time.time() - start) * 1000
            span.set_attribute("retrieval.latency_ms", int(latency_ms))
            span.set_attribute("retrieval.results_count", len(results))
            span_event(span, "retrieve_event", message=(f"Retrieved {len(results)} docs for query {query}"))
            return results

    def generate(self, prompt: str, retrieved: List[Dict[str, Any]]) -> Dict[str, Any]:
        # sama seperti di atas
        # kali ini kita set manual beberapa atribut
        # spesifik generation
        with self.tracer.start_as_current_span("model.generate") as span:
            span.set_attribute("component", "generation")
            span.set_attribute("gen.ai.model.name", self.llm.model_name)
            span.set_attribute("gen.ai.provider", "local-dummy")
            span.set_attribute("gen.ai.input.prompt", prompt)
            # karena RAG menggunakan dokumen hasil retrieval, kita ikutkan juga
            span.set_attribute("gen.ai.context.doc_ids", ",".join(d["id"] for d in retrieved))

            # pura-pura generate dengan LLM dummy
            start = time.time()
            out = self.llm.generate(prompt)
            latency_ms = (time.time() - start) * 1000
            # di production, ini kurang baik. Kalau terlalu panjang,
            # sebaiknya teksnya disimpan, lalu alamatnya / referensinya saja yang dioutput
            span.set_attribute("gen.ai.output.text", out["text"])
            span.set_attribute("gen.ai.output.tokens", int(out["tokens"]))
            span.set_attribute("gen.ai.latency_ms", int(latency_ms))
            span_event(span, "generate_event", message=(f"Model generated {out['tokens']} tokens using model {out['model']}"))
            return out

    def answer(self, user_query: str) -> Dict[str, Any]:
        # span tingkat paling atas
        with self.tracer.start_as_current_span("rag.request") as span:
            span.set_attribute("component", "rag")
            span.set_attribute("rag.user_query", user_query)

            # retrieval, sederhana saja, langsung pakai query user
            retrieved = self.retrieve(user_query, k=3)

            # prompt sederhana
            prompt_parts = [f"User: {user_query}", "Context:"]
            for d in retrieved:
                prompt_parts.append(f"- {d['id']}: {d['text']}")
            prompt = "\n".join(prompt_parts)

            # generate jawaban dengan prompt berisi pertanyaan user
            # dan dokumen hasil retrieval
            gen_out = self.generate(prompt, retrieved)

            # format jadi output yang cantik
            answer = {
                "query": user_query,
                "answer": gen_out["text"],
                "sources": [d["id"] for d in retrieved],
                "llm": gen_out["model"],
            }
            span.set_attribute("rag.answer.source_count", len(answer["sources"]))
            span.set_attribute("rag.status", "ok")
            span_event(span, "final_answer_event", message=(f"RAG answered query with {len(answer['sources'])} sources"))
            return answer


if __name__ == "__main__":
    store = DummyVectorStore()
    llm = DummyLLM()
    rag = DummyRAG(store, llm)

    # pura pura ingest dokumen baru
    new_docs = [
        {"id": "doc4", "text": "Contoh dokumen tambahan tentang konservasi air."},
    ]
    rag.ingest(new_docs)

    # query
    queries = [
        "penurunan tanah jakarta solusi",
        "bagaimana mengurangi penggunaan air tanah?",
    ]

    for q in queries:
        logger.info("--- memanggil rag dengan query: %s ---", q)
        result = rag.answer(q)
        logger.info("Jawaban: %s", result["answer"])
        logger.info("Sumber: %s", result["sources"])


{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "{\"event\": \"ingest_event\", \"message\": \"Ingesting 1 documents\"}", "logger": "dummy_rag", "trace_id": "4ce05f85bc16c0fda98cddc2bd5c44b0", "span_id": "ca079e293b86abc7"}
INFO:dummy_rag:{"event": "ingest_event", "message": "Ingesting 1 documents"}


{
    "name": "ingest_documents",
    "context": {
        "trace_id": "0x4ce05f85bc16c0fda98cddc2bd5c44b0",
        "span_id": "0xca079e293b86abc7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-01T11:22:01.053052Z",
    "end_time": "2026-04-01T11:22:01.076949Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "component": "ingest",
        "document.count": 1,
        "ingest.status": "ok"
    },
    "events": [
        {
            "name": "ingest_event",
            "timestamp": "2026-04-01T11:22:01.053126Z",
            "attributes": {
                "message": "Ingesting 1 documents"
            }
        }
    ],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.38.0",
            "service.name": "dummy-rag-service"
        },


{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "--- memanggil rag dengan query: penurunan tanah jakarta solusi ---", "logger": "dummy_rag"}
INFO:dummy_rag:--- memanggil rag dengan query: penurunan tanah jakarta solusi ---
{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "{\"event\": \"retrieve_event\", \"message\": \"Retrieved 3 docs for query penurunan tanah jakarta solusi\"}", "logger": "dummy_rag", "trace_id": "f2827f2ed2a34095d63400441b8db478", "span_id": "a1415a9d3db801de"}
INFO:dummy_rag:{"event": "retrieve_event", "message": "Retrieved 3 docs for query penurunan tanah jakarta solusi"}


{
    "name": "retrieve_documents",
    "context": {
        "trace_id": "0xf2827f2ed2a34095d63400441b8db478",
        "span_id": "0xa1415a9d3db801de",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2efd2940ef430b51",
    "start_time": "2026-04-01T11:22:01.098248Z",
    "end_time": "2026-04-01T11:22:01.101461Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "component": "retrieval",
        "retrieval.query": "penurunan tanah jakarta solusi",
        "retrieval.top_k": 3,
        "retrieval.latency_ms": 0,
        "retrieval.results_count": 3
    },
    "events": [
        {
            "name": "retrieve_event",
            "timestamp": "2026-04-01T11:22:01.098336Z",
            "attributes": {
                "message": "Retrieved 3 docs for query penurunan tanah jakarta solusi"
            }
        }
    ],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "pyt

{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "{\"event\": \"generate_event\", \"message\": \"Model generated 49 tokens using model dummy-llm-0.1\"}", "logger": "dummy_rag", "trace_id": "f2827f2ed2a34095d63400441b8db478", "span_id": "f6007b3f85848fdb"}
INFO:dummy_rag:{"event": "generate_event", "message": "Model generated 49 tokens using model dummy-llm-0.1"}


...

{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "{\"event\": \"generate_event\", \"message\": \"Model generated 50 tokens using model dummy-llm-0.1\"}", "logger": "dummy_rag", "trace_id": "5ddc07dc0359dcc46226a0eb8ab3d915", "span_id": "55d473677cdaff02"}
INFO:dummy_rag:{"event": "generate_event", "message": "Model generated 50 tokens using model dummy-llm-0.1"}


{
    "name": "model.generate",
    "context": {
        "trace_id": "0x5ddc07dc0359dcc46226a0eb8ab3d915",
        "span_id": "0x55d473677cdaff02",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc06c8371cb7b07e3",
    "start_time": "2026-04-01T11:22:01.156502Z",
    "end_time": "2026-04-01T11:22:01.162670Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "component": "generation",
        "gen.ai.model.name": "dummy-llm-0.1",
        "gen.ai.provider": "local-dummy",
        "gen.ai.input.prompt": "User: bagaimana mengurangi penggunaan air tanah?\nContext:\n- doc3: Kebijakan publik diperlukan untuk mengatur pemompaan air tanah.\n- doc1: Jakarta mengalami penurunan tanah karena overextraction of groundwater.\n- doc2: Solusi teknis termasuk managed aquifer recharge dan piping dari sumber permukaan.",
        "gen.ai.context.doc_ids": "doc3,doc1,doc2",
        "gen.ai.output.text": "[DUMMY GENERATION based on prom

{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "{\"event\": \"final_answer_event\", \"message\": \"RAG answered query with 3 sources\"}", "logger": "dummy_rag", "trace_id": "5ddc07dc0359dcc46226a0eb8ab3d915", "span_id": "c06c8371cb7b07e3"}
INFO:dummy_rag:{"event": "final_answer_event", "message": "RAG answered query with 3 sources"}


{
    "name": "rag.request",
    "context": {
        "trace_id": "0x5ddc07dc0359dcc46226a0eb8ab3d915",
        "span_id": "0xc06c8371cb7b07e3",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-01T11:22:01.148240Z",
    "end_time": "2026-04-01T11:22:01.182788Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "component": "rag",
        "rag.user_query": "bagaimana mengurangi penggunaan air tanah?",
        "rag.answer.source_count": 3,
        "rag.status": "ok"
    },
    "events": [
        {
            "name": "final_answer_event",
            "timestamp": "2026-04-01T11:22:01.170217Z",
            "attributes": {
                "message": "RAG answered query with 3 sources"
            }
        }
    ],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telem

{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "Jawaban: [DUMMY GENERATION based on prompt snippet] User: bagaimana mengurangi penggunaan air tanah?\nContext:\n- doc3: Kebijakan publik diperlukan untuk mengatur pemompaan air tanah.\n- doc1: Jakarta mengalami penurunan tanah karena overextraction of gro", "logger": "dummy_rag"}
INFO:dummy_rag:Jawaban: [DUMMY GENERATION based on prompt snippet] User: bagaimana mengurangi penggunaan air tanah?
Context:
- doc3: Kebijakan publik diperlukan untuk mengatur pemompaan air tanah.
- doc1: Jakarta mengalami penurunan tanah karena overextraction of gro
{"timestamp": "2026-04-01T11:22:01", "level": "INFO", "message": "Sumber: ['doc3', 'doc1', 'doc2']", "logger": "dummy_rag"}
INFO:dummy_rag:Sumber: ['doc3', 'doc1', 'doc2']


In [3]:
pip install langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.40.0
    Uninstalling opentelemetry-api-1.40.0:
      Successfully uninstalled opentelemetry-api-1.40.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.61b0
    Uninstalling opentelemetry-semantic-conventions-0.61b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.61b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-instrumentation 0.61b0 requires opentelemetry-semantic-conventions==0.61b0, but you have opentelemetry-sem

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

import langfuse
from langfuse import observe
from langfuse.langchain import CallbackHandler

client = langfuse.get_client()
langfuse_handler = CallbackHandler()

# berbeda dengan langsung menggunakan API
# langchain memudahkan definisi tool
# langsung pada implementasinya
# menggunakan docstring dan dekorator

@tool
def check_delivery_status(order_id: str):
    """Check the delivery status based on order ID."""
    print(f"Tool check_delivery_status was called with order_id {order_id}")
    return {
        "order_id": order_id,
        "order_status": "In delivery",
        "order_last_location": "Sedang disortir di DC cakung",
        "ETA": "31-12-2025",
    }


@tool
def check_shopping_cart(user_id: str):
    """Retrieve the user's shopping cart based on user ID."""
    print(f"Tool check_shopping_cart was called with user_id {user_id}")
    return {
        "user_id": user_id,
        "items_in_cart": [
            {
                "item_id": 1,
                "item_name": "babibas running shoes",
                "item_type": "shoes",
                "item_details": {"size": 48, "color": "bright pink"},
            },
            {
                "item_id": 22,
                "item_name": "bortusten performance socks",
                "item_type": "socks",
                "item_details": {"size": "XL", "color": "bright yellow"},
            },
            {
                "item_id": 111,
                "item_name": "becs anti-slip shoe lace",
                "item_type": "shoe_accessories",
                "item_details": {"color": "cyan"},
            },
        ],
    }


SYSTEM_PROMPT = """\
Act as a friendly shopping assistant. You will be provided with tools to help you answer the user's queries.
When a tool is not available to help a user's request, suggest to contact a human instead.
Do not make up answers, if a user does not provide the required information, ask them.
Answer should be in fully natural language. It's okay to add small formatting.
Answer in Indonesian by default unless the user asks in english.
"""


# ganti inisiasi kalau menggunakan provider lain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("API_KEY_LLM")
)
# tidak perlu bingung dulu dengan terminologi "Agent" di sini
# anggap saja LLM biasa dengan tool
agent = create_agent(
    model=llm,
    tools=[check_delivery_status, check_shopping_cart],
    system_prompt=SYSTEM_PROMPT,
)


# By default menggunakan gemini. Ganti nama model kalau menggunakan provider lain
@observe
def chat(query):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config={"callbacks": [langfuse_handler]},
    )
    return result["messages"][-1].content


print(">> Pemanggilan 1, perlu tool check status pengiriman <<")
print(
    chat(
        "Halo, saya ingin mengecek status pengiriman pesanan saya dengan ID 1112231233"
    )
)
print("====== separator ======")
print(">> Pemanggilan 2, perlu tool check keranjang belanja <<")
print(
    chat("Halo, saya ingin mengecek keranjang belanja saya dengan user ID Ivan#19827")
)
print("====== separator ======")
print(">> Pemanggilan 3, tidak perlu tool apapun, cuma pertanyaan biasa <<")
print(chat("Halo, apa kabar? Siapa kamu?"))

>> Pemanggilan 1, perlu tool check status pengiriman <<
Tool check_delivery_status was called with order_id 1112231233


Tentu, saya bisa bantu!

Pesanan Anda dengan ID **1112231233** saat ini berstatus **Dalam Pengiriman** (In delivery).
Lokasi terakhir pesanan Anda adalah **Sedang disortir di DC Cakung**.
Estimasi waktu kedatangan (ETA) adalah **31-12-2025**.

Apakah ada hal lain yang bisa saya bantu?
====== separator ======
>> Pemanggilan 2, perlu tool check keranjang belanja <<
Tool check_shopping_cart was called with user_id Ivan#19827


Halo Ivan#19827!

Berikut adalah isi keranjang belanja Anda:

*   **Sepatu Lari Babibas**
    *   ID Barang: 1
    *   Warna: Merah muda terang
    *   Ukuran: 48
*   **Kaus Kaki Performa Bortusten**
    *   ID Barang: 22
    *   Warna: Kuning terang
    *   Ukuran: XL
*   **Tali Sepatu Anti-Selip Becs**
    *   ID Barang: 111
    *   Warna: Cyan

Apakah ada hal lain yang bisa saya bantu?
====== separator ======
>> Pemanggilan 3, tidak perlu tool apapun, cuma pertanyaan biasa <<
Halo! Saya adalah asisten belanja virtual Anda. Saya di sini untuk membantu Anda dengan pertanyaan terkait belanja. Ada yang bisa saya bantu?
